# Leela Chess Zero BT4 · quickstart

<a href="https://colab.research.google.com/github/chessformer-lens/chessformer_lens/blob/main/notebooks/leela_quickstart.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"></a>

Look inside Leela Chess Zero's BT4 transformer (185M parameters, 15 layers × 32 heads) with
[chessformer_lens](https://github.com/chessformer-lens/chessformer_lens): its policy, its attention,
and which heads carry a move.

**Runtime:** pick a GPU (*Runtime → Change runtime type → T4 GPU*). Every cell runs on CPU too, just
slower; the one marked *heavy* needs a GPU to finish in reasonable time.

In [ ]:
# Installs from GitHub until the next PyPI release (Leela needs nothing else)
!pip install -q git+https://github.com/chessformer-lens/chessformer_lens

## Load BT4
`bt4` is the base BT4 net; `bt4-332` is its policytune-332 fine-tune, the net lc0 sends to engine
competitions. The first load downloads the weights from lczero.org (~336 MB) and checks them.

In [ ]:
import chess
from chessformer_lens import LeelaEngine
import chessformer_lens.interp_plot as ip
import chessformer_lens.interp_widget as iw

eng = LeelaEngine("bt4")
ip.set_theme("light")
print(f"{eng.cfg.num_blocks} layers × {eng.cfg.num_heads} heads × {eng.cfg.dim_vit}d on {eng.device}")

## A position
The Opera game (Morphy, 1858), just before the famous queen sacrifice. Leela has no rating input, so
the rating argument the methods take is ignored.

In [ ]:
board = chess.Board("4kb1r/p2n1ppp/4q3/4p1B1/4P3/1Q6/PPP2PPP/2KR4 w k - 0 16")
res = eng.evaluate(board, 0)
print("top moves:", [(board.san(chess.Move.from_uci(u)), round(p, 3)) for u, p in res["policy"][:5]])
print("win/draw/loss:", {k: round(v, 3) for k, v in res["wdl"].items()}, "· moves left:", round(res["mlh"]))
ip.plot_position(eng, board)

## When does the move form?
The logit lens reads the top move off the residual stream after every attention and MLP
sub-layer: where it takes over is where the network "decides".

In [ ]:
top = res["policy"][0][0]
ip.plot_logit_curve(eng, board, ucis=[top, res["policy"][1][0]])

## Attention, live
Pick a layer and a head, click a square to set the query. *Semantic* is QKᵀ, *geometric* is the
generated smolgen bias (GAB), and the last board is the attention the head actually runs.

In [ ]:
iw.attention_widget(eng, board, layer=10, head=5)

## Which heads carry the move *(heavy)*
Every one of the 480 heads is ablated in turn (its exact write removed) and the change in the
move's logit recorded: blue heads carry it, orange heads suppress it. ~500 forward passes, so use
a GPU.

In [ ]:
ip.plot_carrier_heads(eng, board, uci=top)

## The whole app, in a new tab
Every panel above and more (the move microscope, neurons) in the app, on the engine you already
loaded. In Colab it opens in a new window through Colab's port proxy.

In [ ]:
from chessformer_lens import web
app = web.serve(engine=eng)